# Model Evaluation: YOLOv8n + CBAM (3-Layer)
**Experiment ID:** `cbam_3layer`  
**Architecture:** YOLOv8n with CBAM inserted after SPPF, P4-head, P5-head  
**Dataset:** Helmet Detection (2 classes: helmet, non-helmet)  
> Run this script after training. Point `WEIGHTS` to your `best.pt`.

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
# ─────────────────────────────────────────────
#  CONFIGURATION  ←  Edit only this cell
# ─────────────────────────────────────────────
EXPERIMENT_ID = "cbam_3layer"
WEIGHTS       = r"c:/Users/soumy/OneDrive/Desktop/Testing_Scripts(final)/models/best (3_Layer).pt"
DATA_YAML     = r"c:/Users/soumy/OneDrive/Desktop/Testing_Scripts(final)/data_big.yaml"
DATASET_ROOT  = r"C:/Users/soumy/OneDrive/Desktop/test(big)"
IMGSZ         = 640
BATCH         = 16
CONF_THRES    = 0.001
IOU_THRES     = 0.5
OUTPUT_DIR    = rf"c:/Users/soumy/OneDrive/Desktop/Testing_Scripts(final)/results_small/eval_{EXPERIMENT_ID}"
CLASS_NAMES   = ["helmet", "non-helmet"]
# ─────────────────────────────────────────────

In [2]:
!pip install ultralytics -q

In [3]:
# ── CBAM must be registered before loading weights ──
import torch, torch.nn as nn, sys

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.pool_h = nn.AdaptiveAvgPool2d(1)
        self.pool_m = nn.AdaptiveMaxPool2d(1)
        r = max(channels // reduction, 1)
        self.fc = nn.Sequential(nn.Conv2d(channels, r, 1, bias=False), nn.ReLU(inplace=True), nn.Conv2d(r, channels, 1, bias=False))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return x * self.sigmoid(self.fc(self.pool_h(x)) + self.fc(self.pool_m(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return x * self.sigmoid(self.conv(torch.cat([torch.mean(x,1,True), torch.max(x,1,True)[0]], 1)))

class CBAM(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.kernel_size = kernel_size
        self.ca = None; self.sa = None
    def forward(self, x):
        if self.ca is None:
            self.ca = ChannelAttention(x.shape[1]).to(x.device)
            self.sa = SpatialAttention(self.kernel_size).to(x.device)
        x = self.ca(x)
        return x * self.sa(x)

import ultralytics.nn.tasks as tasks, ultralytics.nn.modules as modules
for cls in [CBAM, ChannelAttention, SpatialAttention]:
    setattr(tasks, cls.__name__, cls)
    setattr(modules, cls.__name__, cls)
    for s in ("block","conv","head"):
        if hasattr(modules,s): setattr(getattr(modules,s), cls.__name__, cls)
    sys.modules['__main__'].__dict__[cls.__name__] = cls
torch.serialization.add_safe_globals([CBAM, ChannelAttention, SpatialAttention])
print("[✓] CBAM registered — safe to load weights.")

[✓] CBAM registered — safe to load weights.


In [4]:
import os, json, csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from ultralytics import YOLO

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"[✓] Output directory: {OUTPUT_DIR}")
print(f"[✓] Loading weights : {WEIGHTS}")

[✓] Output directory: c:/Users/soumy/OneDrive/Desktop/Testing_Scripts(final)/results_small/eval_cbam_3layer
[✓] Loading weights : c:/Users/soumy/OneDrive/Desktop/Testing_Scripts(final)/models/best (3_Layer).pt


## 1 · Validation on Val Split

In [5]:
model = YOLO(WEIGHTS)

val_results = model.val(
    data=DATA_YAML,
    split="val",
    imgsz=IMGSZ,
    batch=BATCH,
    conf=CONF_THRES,
    iou=IOU_THRES,
    save_json=True,
    plots=True,
    project=OUTPUT_DIR,
    name="val",
    exist_ok=True, workers=0,
)

print("\n[Val] Raw metrics object:", val_results)

Ultralytics 8.4.6  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)


cbam summary (fused): 97 layers, 3,024,764 parameters, 0 gradients, 8.1 GFLOPs


val: Fast image access  (ping: 0.10.0 ms, read: 357.1124.1 MB/s, size: 53.4 KB)



val: Scanning C:\Users\soumy\OneDrive\Desktop\test(big)\labels.cache... 1766 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1766/1766 205.8Mit/s 0.0s


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 1% ──────────── 1/111 1.4s/it 0.4s<2:33

...

            non-helmet        339       1803      0.931       0.95      0.965      0.699


Speed: 0.4ms preprocess, 4.2ms inference, 0.0ms loss, 1.4ms postprocess per image


Saving C:\Users\soumy\OneDrive\Desktop\Testing_Scripts(final)\results_small\eval_cbam_3layer\val\predictions.json...


Results saved to C:\Users\soumy\OneDrive\Desktop\Testing_Scripts(final)\results_small\eval_cbam_3layer\val



[Val] Raw metrics object: ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002A6F4B04C20>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,

## 2 · Evaluation on Test Split

In [6]:
test_results = model.val(
    data=DATA_YAML,
    split="test",
    imgsz=IMGSZ,
    batch=BATCH,
    conf=CONF_THRES,
    iou=IOU_THRES,
    save_json=True,
    plots=True,
    project=OUTPUT_DIR,
    name="test",
    exist_ok=True, workers=0,
)

print("\n[Test] Raw metrics object:", test_results)

Ultralytics 8.4.6  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)


val: Fast image access  (ping: 0.10.1 ms, read: 150.0117.1 MB/s, size: 32.2 KB)



val: Scanning C:\Users\soumy\OneDrive\Desktop\test(big)\labels.cache... 1766 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1766/1766 370.4Mit/s 0.0s


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 1% ──────────── 1/111 1.3s/it 0.4s<2:23


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 2% ──────────── 2/111 1.2it/s 0.8s<1:31

...

            non-helmet        339       1803      0.931       0.95      0.965      0.699


Speed: 0.4ms preprocess, 4.0ms inference, 0.0ms loss, 1.4ms postprocess per image


Saving C:\Users\soumy\OneDrive\Desktop\Testing_Scripts(final)\results_small\eval_cbam_3layer\test\predictions.json...


Results saved to C:\Users\soumy\OneDrive\Desktop\Testing_Scripts(final)\results_small\eval_cbam_3layer\test



[Test] Raw metrics object: ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002A687A6FED0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046

## 3 · Extract & Display Metrics

In [7]:
def extract_metrics(results, split_name):
    """Extract a flat dict of metrics from a YOLO val results object."""
    box = results.box
    metrics = {
        "experiment":   EXPERIMENT_ID,
        "split":        split_name,
        # ── overall ──
        "mAP50":        round(float(box.map50),  4),
        "mAP50_95":     round(float(box.map),    4),
        "precision":    round(float(box.mp),     4),
        "recall":       round(float(box.mr),     4),
        "f1":           round(float(np.mean(box.f1)), 4),
    }
    # ── per-class ──
    for i, cls_name in enumerate(CLASS_NAMES):
        try:
            metrics[f"AP50_{cls_name}"]    = round(float(box.ap50[i]),  4)
            metrics[f"AP50_95_{cls_name}"] = round(float(box.ap[i]),    4)
            metrics[f"P_{cls_name}"]       = round(float(box.p[i]),     4)
            metrics[f"R_{cls_name}"]       = round(float(box.r[i]),     4)
            metrics[f"F1_{cls_name}"]      = round(float(box.f1[i]),    4)
        except Exception:
            pass
    return metrics

val_metrics  = extract_metrics(val_results,  "val")
test_metrics = extract_metrics(test_results, "test")

# Pretty print
print("=" * 55)
print(f"  EXPERIMENT : {EXPERIMENT_ID.upper()}")
print("=" * 55)
for split, m in [("VAL", val_metrics), ("TEST", test_metrics)]:
    print(f"\n  [{split}]")
    print(f"    mAP@50        : {m['mAP50']}")
    print(f"    mAP@50-95     : {m['mAP50_95']}")
    print(f"    Precision     : {m['precision']}")
    print(f"    Recall        : {m['recall']}")
    print(f"    F1            : {m['f1']}")
    for cls in CLASS_NAMES:
        print(f"    AP50 [{cls:<12}]: {m.get(f'AP50_{cls}', 'N/A')}")
print("=" * 55)

  EXPERIMENT : CBAM_3LAYER

  [VAL]
    mAP@50        : 0.9727
    mAP@50-95     : 0.705
    Precision     : 0.9464
    Recall        : 0.9482
    F1            : 0.9472
    AP50 [helmet      ]: 0.98
    AP50 [non-helmet  ]: 0.9653

  [TEST]
    mAP@50        : 0.9727
    mAP@50-95     : 0.705
    Precision     : 0.9464
    Recall        : 0.9482
    F1            : 0.9472
    AP50 [helmet      ]: 0.98
    AP50 [non-helmet  ]: 0.9653


## 4 · Save Metrics to CSV & JSON

In [8]:
all_metrics = [val_metrics, test_metrics]

# CSV  (append-friendly for combining across experiments later)
csv_path = f"{OUTPUT_DIR}/metrics_{EXPERIMENT_ID}.csv"
df = pd.DataFrame(all_metrics)
df.to_csv(csv_path, index=False)
print(f"[✓] CSV saved  → {csv_path}")

# JSON (full record)
json_path = f"{OUTPUT_DIR}/metrics_{EXPERIMENT_ID}.json"
with open(json_path, "w") as f:
    json.dump(all_metrics, f, indent=2)
print(f"[✓] JSON saved → {json_path}")

df

[✓] CSV saved  → c:/Users/soumy/OneDrive/Desktop/Testing_Scripts(final)/results_small/eval_cbam_3layer/metrics_cbam_3layer.csv
[✓] JSON saved → c:/Users/soumy/OneDrive/Desktop/Testing_Scripts(final)/results_small/eval_cbam_3layer/metrics_cbam_3layer.json


,experiment,split,mAP50,mAP50_95,precision,recall,f1,AP50_helmet,AP50_95_helmet,P_helmet,R_helmet,F1_helmet,AP50_non-helmet,AP50_95_non-helmet,P_non-helmet,R_non-helmet,F1_non-helmet
0,cbam_3layer,val,0.9727,0.705,0.9464,0.9482,0.9472,0.98,0.711,0.9614,0.9463,0.9538,0.9653,0.699,0.9315,0.95,0.9407
1,cbam_3layer,test,0.9727,0.705,0.9464,0.9482,0.9472,0.98,0.711,0.9614,0.9463,0.9538,0.9653,0.699,0.9315,0.95,0.9407


## 5 · Metrics Bar Chart

In [9]:
metric_keys   = ["mAP50", "mAP50_95", "precision", "recall", "f1"]
metric_labels = ["mAP@50", "mAP@50-95", "Precision", "Recall", "F1"]

val_vals  = [val_metrics[k]  for k in metric_keys]
test_vals = [test_metrics[k] for k in metric_keys]

x     = np.arange(len(metric_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, val_vals,  width, label="Val",  color="#4C72B0", alpha=0.88)
bars2 = ax.bar(x + width/2, test_vals, width, label="Test", color="#DD8452", alpha=0.88)

ax.set_ylim(0, 1.05)
ax.set_xticks(x)
ax.set_xticklabels(metric_labels, fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title(f"Evaluation Metrics — {EXPERIMENT_ID.upper()}", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(axis="y", linestyle="--", alpha=0.5)

for bar in [*bars1, *bars2]:
    h = bar.get_height()
    ax.annotate(f"{h:.3f}", xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8)

plt.tight_layout()
chart_path = f"{OUTPUT_DIR}/metrics_chart_{EXPERIMENT_ID}.png"
plt.savefig(chart_path, dpi=150)
plt.show()
print(f"[✓] Chart saved → {chart_path}")

<Figure size 1000x500 with 1 Axes>

[✓] Chart saved → c:/Users/soumy/OneDrive/Desktop/Testing_Scripts(final)/results_small/eval_cbam_3layer/metrics_chart_cbam_3layer.png


## 6 · Per-Class AP Bar Chart

In [10]:
ap50_vals = [test_metrics.get(f"AP50_{c}", 0) for c in CLASS_NAMES]
ap95_vals = [test_metrics.get(f"AP50_95_{c}", 0) for c in CLASS_NAMES]

x     = np.arange(len(CLASS_NAMES))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - width/2, ap50_vals, width, label="AP@50",    color="#55A868", alpha=0.88)
b2 = ax.bar(x + width/2, ap95_vals, width, label="AP@50-95", color="#C44E52", alpha=0.88)

ax.set_ylim(0, 1.05)
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, fontsize=12)
ax.set_ylabel("AP Score", fontsize=12)
ax.set_title(f"Per-Class AP (Test) — {EXPERIMENT_ID.upper()}", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(axis="y", linestyle="--", alpha=0.5)

for bar in [*b1, *b2]:
    h = bar.get_height()
    ax.annotate(f"{h:.3f}", xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 3), textcoords="offset points", ha="center", fontsize=9)

plt.tight_layout()
pc_path = f"{OUTPUT_DIR}/per_class_ap_{EXPERIMENT_ID}.png"
plt.savefig(pc_path, dpi=150)
plt.show()
print(f"[✓] Per-class chart saved → {pc_path}")

<Figure size 800x500 with 1 Axes>

[✓] Per-class chart saved → c:/Users/soumy/OneDrive/Desktop/Testing_Scripts(final)/results_small/eval_cbam_3layer/per_class_ap_cbam_3layer.png


## 7 · Inference Speed

In [11]:
import time, glob, random

test_images = glob.glob(f"{DATASET_ROOT}/images/**/*.jpg", recursive=True)
test_images += glob.glob(f"{DATASET_ROOT}/images/**/*.png", recursive=True)

WARMUP = min(10, max(0, len(test_images) - 1))
MEASURE = min(200, len(test_images) - WARMUP)
sample  = random.sample(test_images, WARMUP + MEASURE)

# Warmup
for img in sample[:WARMUP]:
    model(img, verbose=False)

# Timed
times = []
for img in sample[WARMUP:]:
    t0 = time.perf_counter()
    model(img, verbose=False, imgsz=IMGSZ)
    times.append((time.perf_counter() - t0) * 1000)

speed_metrics = {
    "mean_ms":   round(np.mean(times),   2),
    "median_ms": round(np.median(times), 2),
    "std_ms":    round(np.std(times),    2),
    "fps":       round(1000 / np.mean(times), 2),
    "n_images":  MEASURE,
}

print(f"  Mean latency : {speed_metrics['mean_ms']} ms")
print(f"  Median       : {speed_metrics['median_ms']} ms")
print(f"  Std dev      : {speed_metrics['std_ms']} ms")
print(f"  FPS          : {speed_metrics['fps']}")

# Append to JSON
with open(json_path) as f:
    full = json.load(f)
full.append({"experiment": EXPERIMENT_ID, "split": "speed", **speed_metrics})
with open(json_path, "w") as f:
    json.dump(full, f, indent=2)

  Mean latency : 21.04 ms
  Median       : 16.17 ms
  Std dev      : 17.19 ms
  FPS          : 47.52


## 8 · Qualitative Prediction Samples (Test Set)

In [12]:
import cv2
from PIL import Image

COLORS = {"helmet": (0, 200, 80), "non-helmet": (220, 50, 50)}
N_SHOW = 9
sample_imgs = random.sample(test_images, N_SHOW)

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
axes = axes.flatten()

for ax, img_path in zip(axes, sample_imgs):
    img_bgr = cv2.imread(img_path)
    preds   = model(img_path, imgsz=IMGSZ, conf=0.25, verbose=False)[0]
    
    for box in preds.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cls_id  = int(box.cls[0])
        conf    = float(box.conf[0])
        label   = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else str(cls_id)
        color   = COLORS.get(label, (200, 200, 200))
        cv2.rectangle(img_bgr, (x1, y1), (x2, y2), color, 2)
        cv2.putText(img_bgr, f"{label} {conf:.2f}", (x1, max(y1-6, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
    
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    ax.axis("off")
    ax.set_title(Path(img_path).name[:30], fontsize=8)

fig.suptitle(f"Sample Predictions — {EXPERIMENT_ID.upper()}", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
samples_path = f"{OUTPUT_DIR}/sample_predictions_{EXPERIMENT_ID}.png"
plt.savefig(samples_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"[✓] Sample grid saved → {samples_path}")

<Figure size 1500x1500 with 9 Axes>

[✓] Sample grid saved → c:/Users/soumy/OneDrive/Desktop/Testing_Scripts(final)/results_small/eval_cbam_3layer/sample_predictions_cbam_3layer.png


## 9 · Model Info (Parameters & FLOPs)

In [13]:
info = model.info(verbose=True)
print(f"\n[✓] Model info printed above.")
print(f"    Weights file: {WEIGHTS}")

cbam summary (fused): 97 layers, 3,024,764 parameters, 0 gradients, 8.1 GFLOPs



[✓] Model info printed above.
    Weights file: c:/Users/soumy/OneDrive/Desktop/Testing_Scripts(final)/models/best (3_Layer).pt


## 10 · Final Summary

In [14]:
print("\n" + "═" * 60)
print(f"  FINAL SUMMARY — {EXPERIMENT_ID.upper()}")
print("═" * 60)
print(f"  {'Metric':<20} {'Val':>10} {'Test':>10}")
print("  " + "-" * 42)
for key, label in zip(metric_keys, metric_labels):
    print(f"  {label:<20} {val_metrics[key]:>10.4f} {test_metrics[key]:>10.4f}")
print("  " + "-" * 42)
print(f"  {'FPS (mean)':<20} {'':>10} {speed_metrics['fps']:>10.2f}")
print("═" * 60)
print(f"\n  Outputs saved to: {OUTPUT_DIR}/")


════════════════════════════════════════════════════════════
  FINAL SUMMARY — CBAM_3LAYER
════════════════════════════════════════════════════════════
  Metric                      Val       Test
  ------------------------------------------
  mAP@50                   0.9727     0.9727
  mAP@50-95                0.7050     0.7050
  Precision                0.9464     0.9464
  Recall                   0.9482     0.9482
  F1                       0.9472     0.9472
  ------------------------------------------
  FPS (mean)                           47.52
════════════════════════════════════════════════════════════

  Outputs saved to: c:/Users/soumy/OneDrive/Desktop/Testing_Scripts(final)/results_small/eval_cbam_3layer/
